# Plot 3: CARROT vs Routerbench router on Routerbench

Compares CARROT-KNN and CARROT-RoBERTa against the Routerbench routing rule applied to the same accuracy predictors.

- **CARROT rule**: `argmax_m (1-λ)·score_pred − λ·cost_pred`  (per-prompt predicted cost)
- **Routerbench rule**: `argmax_m (1-λ)·score_pred − λ·avg_cost`  (static per-model mean cost)

In [ ]:
import os, sys
sys.path.insert(0, '../carrot/')
import numpy as np
import matplotlib.pyplot as plt
from utils import route, route_routerbench

os.makedirs('../plots', exist_ok=True)
PREDS_DIR = '../data/routerbench/preds'

meta = np.load(f'{PREDS_DIR}/meta.npy', allow_pickle=True).item()
models = meta['models']
Y_test = meta['Y_test']
C_test = meta['C_test']
print(f'{len(models)} models; test n={Y_test.shape[0]}')

In [ ]:
def load(name):
    path = f'{PREDS_DIR}/{name}.npy'
    if not os.path.exists(path):
        print(f'MISSING: {path}')
        return None
    return np.load(path, allow_pickle=True)

Y_hat = {m: load(f'Y_hat_{m}') for m in ['carrot-knn', 'carrot-roberta']}
C_hat = {m: load(f'C_hat_{m}') for m in ['carrot-knn', 'carrot-roberta']}

In [ ]:
mult = 100
curves = {}

for name in ['carrot-knn', 'carrot-roberta']:
    if Y_hat[name] is None or C_hat[name] is None:
        continue
    # CARROT rule: per-prompt predicted cost
    c, p = route(Y_hat[name], C_test, mult * C_hat[name], Y_test)
    curves[f'carrot-{name.split("-")[-1]}'] = (c, p)
    # Routerbench rule: static per-model avg cost, same score predictions
    c, p = route_routerbench(Y_hat[name], C_test, Y_test)
    curves[f'rb-{name.split("-")[-1]}'] = (c, p)

print('Curves produced:', list(curves.keys()))

In [ ]:
from matplotlib.ticker import MaxNLocator

labels = {'carrot-knn': 'CARROT (KNN)', 'carrot-roberta': 'CARROT (RoBERTa)',
          'rb-knn': 'Routerbench (KNN)', 'rb-roberta': 'Routerbench (RoBERTa)'}
colors = {'carrot-knn': 'red', 'carrot-roberta': 'orange',
          'rb-knn': 'purple', 'rb-roberta': 'blue'}
markers = ['o', 's', 'D', '^', 'v', 'p', '*', 'x', '+', 'h', 'H', 'd', '>', 'P']

LABEL_MODELS = {'gpt-4-1106-preview', 'zero-one-ai/Yi-34B-Chat',
                'gpt-3.5-turbo-1106', 'mistralai/mixtral-8x7b-chat'}

fig, ax = plt.subplots(1, 1, figsize=(4.3, 4.3))
for name, (c, p) in curves.items():
    linestyle = '--' if name.startswith('carrot') else '-'
    ax.errorbar(c, p, c=colors[name], linestyle=linestyle, linewidth=1, label=labels[name])

for i, m in enumerate(models):
    x, y = C_test[:, i].mean(0), Y_test[:, i].mean(0)
    ax.scatter([x], [y], marker=markers[i % len(markers)])
    if m in LABEL_MODELS:
        ax.annotate(m.split('/')[-1], (x, y), size=6)

ax.set_title('Routerbench — CARROT vs Routerbench router')
ax.set_xlabel('Cost Per Query, $')
ax.set_ylabel('Accuracy')
ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
ax.legend()
ax.grid(True)
fig.savefig('../plots/routerbench_vs_rb.pdf', bbox_inches='tight')
plt.show()